# FashionMNIST Single-Branch ANN on Colab T4

Notebook này tự chạy độc lập trên Colab, không phụ thuộc việc repo có nằm sẵn trong Google Drive hay không. Nếu mount được Drive thì artifact sẽ được lưu vào Drive, còn không thì lưu vào `/content/artifacts`.

In [3]:
import os
from pathlib import Path

IN_COLAB = 'COLAB_GPU' in os.environ
save_root = Path('/content')

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    preferred_roots = [
        Path('/content/drive/MyDrive/Lab3-LT'),
        Path('/content/drive/MyDrive/Colab Notebooks/Lab3-LT'),
        Path('/content/drive/MyDrive')
    ]
    for candidate in preferred_roots:
        if candidate.exists():
            save_root = candidate
            break

artifact_dir = save_root / 'artifacts' / 'fashion_mnist_ann_single_branch'
cache_dir = save_root / 'data' / 'fashion_mnist_ann_single_branch_cache'
artifact_dir.mkdir(parents=True, exist_ok=True)
cache_dir.mkdir(parents=True, exist_ok=True)

print('IN_COLAB =', IN_COLAB)
print('save_root =', save_root)
print('artifact_dir =', artifact_dir)
print('cache_dir =', cache_dir)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
IN_COLAB = True
save_root = /content/drive/MyDrive
artifact_dir = /content/drive/MyDrive/artifacts/fashion_mnist_ann_single_branch
cache_dir = /content/drive/MyDrive/data/fashion_mnist_ann_single_branch_cache


In [4]:
if IN_COLAB:
    !pip install -q scikit-image opencv-python-headless
    !nvidia-smi

Wed Mar 25 19:08:06 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   42C    P8             13W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [5]:
import json
import math
import random
from dataclasses import dataclass

import cv2
import numpy as np
import torch
import torch.nn as nn
from skimage.feature import hog
from torch.utils.data import DataLoader, Dataset
from torchvision.datasets import FashionMNIST

def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def compute_slant_angle(image: np.ndarray, threshold: float = 0.1) -> float:
    mask = image > threshold
    ys, xs = np.nonzero(mask)
    if len(xs) < 5:
        return 0.0
    weights = image[ys, xs].astype(np.float64)
    if weights.sum() <= 1e-8:
        return 0.0
    x_mean = np.average(xs, weights=weights)
    y_mean = np.average(ys, weights=weights)
    x_centered = xs - x_mean
    y_centered = ys - y_mean
    var_y = np.average(y_centered * y_centered, weights=weights)
    if var_y <= 1e-8:
        return 0.0
    cov_xy = np.average(x_centered * y_centered, weights=weights)
    return math.atan(cov_xy / var_y)

def deslant_image(image: np.ndarray) -> np.ndarray:
    angle = compute_slant_angle(image)
    shear = -math.tan(angle)
    if abs(shear) < 1e-4:
        return image.copy()
    h, w = image.shape
    center_y = (h - 1) / 2.0
    tx = -shear * center_y
    matrix = np.array([[1.0, shear, tx], [0.0, 1.0, 0.0]], dtype=np.float32)
    warped = cv2.warpAffine(image, matrix, (w, h), flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_CONSTANT, borderValue=0.0)
    return np.clip(warped, 0.0, 1.0)

def recenter_image(image: np.ndarray, threshold: float = 0.1) -> np.ndarray:
    mask = image > threshold
    ys, xs = np.nonzero(mask)
    if len(xs) < 5:
        return image.copy()
    weights = image[ys, xs].astype(np.float64)
    if weights.sum() <= 1e-8:
        return image.copy()
    x_mean = np.average(xs, weights=weights)
    y_mean = np.average(ys, weights=weights)
    target_x = (image.shape[1] - 1) / 2.0
    target_y = (image.shape[0] - 1) / 2.0
    shift_x = target_x - x_mean
    shift_y = target_y - y_mean
    matrix = np.array([[1.0, 0.0, shift_x], [0.0, 1.0, shift_y]], dtype=np.float32)
    warped = cv2.warpAffine(image, matrix, (image.shape[1], image.shape[0]), flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_CONSTANT, borderValue=0.0)
    return np.clip(warped, 0.0, 1.0)

def random_affine(image: np.ndarray, max_rotate: float, max_shift: float, max_shear: float) -> np.ndarray:
    angle = random.uniform(-max_rotate, max_rotate)
    shear_deg = random.uniform(-max_shear, max_shear)
    tx = random.uniform(-max_shift, max_shift) * image.shape[1]
    ty = random.uniform(-max_shift, max_shift) * image.shape[0]
    center = ((image.shape[1] - 1) / 2.0, (image.shape[0] - 1) / 2.0)
    rot = cv2.getRotationMatrix2D(center, angle, 1.0)
    rot[0, 2] += tx
    rot[1, 2] += ty
    warped = cv2.warpAffine(image, rot, (image.shape[1], image.shape[0]), flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_CONSTANT, borderValue=0.0)
    shear = math.tan(math.radians(shear_deg))
    center_y = (image.shape[0] - 1) / 2.0
    tx_shear = -shear * center_y
    shear_matrix = np.array([[1.0, shear, tx_shear], [0.0, 1.0, 0.0]], dtype=np.float32)
    warped = cv2.warpAffine(warped, shear_matrix, (image.shape[1], image.shape[0]), flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_CONSTANT, borderValue=0.0)
    return np.clip(warped, 0.0, 1.0)

def extract_hog_features(image: np.ndarray) -> np.ndarray:
    features = hog(image, orientations=9, pixels_per_cell=(4, 4), cells_per_block=(2, 2), block_norm='L2-Hys', transform_sqrt=False, feature_vector=True)
    return features.astype(np.float32)

def preprocess_image(image: np.ndarray, use_augmentation: bool) -> np.ndarray:
    proc = image.astype(np.float32) / 255.0
    if use_augmentation:
        proc = random_affine(proc, max_rotate=10.0, max_shift=0.08, max_shear=8.0)
    proc = deslant_image(proc)
    proc = recenter_image(proc)
    raw = proc.reshape(-1).astype(np.float32)
    hog_feat = extract_hog_features(proc)
    return np.concatenate([raw, hog_feat], axis=0).astype(np.float32)

def load_or_create_cache(split_name: str, images: np.ndarray, labels: np.ndarray, cache_root: Path, augment: bool):
    suffix = 'aug' if augment else 'plain'
    cache_path = cache_root / f'{split_name}_{suffix}.npz'
    if cache_path.exists():
        data = np.load(cache_path)
        return {key: data[key] for key in data.files}
    features = [preprocess_image(img, use_augmentation=augment) for img in images]
    cache = {'features': np.stack(features).astype(np.float32), 'labels': labels.astype(np.int64)}
    np.savez_compressed(cache_path, **cache)
    return cache

class FeatureDataset(Dataset):
    def __init__(self, features: np.ndarray, labels: np.ndarray, mean: np.ndarray, std: np.ndarray) -> None:
        self.features = (features - mean) / std
        self.labels = labels
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, idx):
        return torch.from_numpy(self.features[idx]).float(), torch.tensor(self.labels[idx]).long()

class SingleBranchAnn(nn.Module):
    def __init__(self, input_dim: int, dropout: float = 0.25) -> None:
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 1024), nn.BatchNorm1d(1024), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(1024, 512), nn.BatchNorm1d(512), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(512, 256), nn.BatchNorm1d(256), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(256, 10),
        )
    def forward(self, x):
        return self.net(x)

@dataclass
class EpochResult:
    loss: float
    accuracy: float

def run_epoch(model, loader, criterion, device, optimizer=None):
    train_mode = optimizer is not None
    model.train(train_mode)
    total_loss, total_correct, total_samples = 0.0, 0, 0
    for features, labels in loader:
        features, labels = features.to(device), labels.to(device)
        if optimizer is not None:
            optimizer.zero_grad(set_to_none=True)
        logits = model(features)
        loss = criterion(logits, labels)
        if optimizer is not None:
            loss.backward()
            optimizer.step()
        total_loss += loss.item() * labels.size(0)
        total_correct += (logits.argmax(dim=1) == labels).sum().item()
        total_samples += labels.size(0)
    return EpochResult(loss=total_loss / max(total_samples, 1), accuracy=total_correct / max(total_samples, 1))

def evaluate_test(model, loader, device):
    model.eval()
    total_correct, total_samples = 0, 0
    confusion = np.zeros((10, 10), dtype=np.int64)
    with torch.no_grad():
        for features, labels in loader:
            features, labels = features.to(device), labels.to(device)
            logits = model(features)
            preds = logits.argmax(dim=1)
            total_correct += (preds == labels).sum().item()
            total_samples += labels.size(0)
            for truth, pred in zip(labels.cpu().numpy(), preds.cpu().numpy()):
                confusion[truth, pred] += 1
    per_class_accuracy = {}
    for class_idx in range(10):
        total_in_class = confusion[class_idx].sum()
        acc = float(confusion[class_idx, class_idx] / total_in_class) if total_in_class > 0 else 0.0
        per_class_accuracy[str(class_idx)] = acc
    return total_correct / max(total_samples, 1), confusion, per_class_accuracy

set_seed(42)
data_root = save_root / 'data'
train_ds_raw = FashionMNIST(root=str(data_root), train=True, download=True)
test_ds_raw = FashionMNIST(root=str(data_root), train=False, download=True)
train_cache = load_or_create_cache('train', train_ds_raw.data.numpy(), train_ds_raw.targets.numpy(), cache_dir, augment=True)
test_cache = load_or_create_cache('test', test_ds_raw.data.numpy(), test_ds_raw.targets.numpy(), cache_dir, augment=False)
mean = train_cache['features'].mean(axis=0, dtype=np.float64).astype(np.float32)
std = train_cache['features'].std(axis=0, dtype=np.float64).astype(np.float32)
std = np.where(std < 1e-6, 1.0, std)
train_ds = FeatureDataset(train_cache['features'], train_cache['labels'], mean, std)
test_ds = FeatureDataset(test_cache['features'], test_cache['labels'], mean, std)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device =', device)
train_loader = DataLoader(train_ds, batch_size=256, shuffle=True, num_workers=2, pin_memory=device.type == 'cuda')
test_loader = DataLoader(test_ds, batch_size=256, shuffle=False, num_workers=2, pin_memory=device.type == 'cuda')
model = SingleBranchAnn(input_dim=train_cache['features'].shape[1], dropout=0.25).to(device)
criterion = nn.CrossEntropyLoss(label_smoothing=0.05)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=50)
history = []
for epoch in range(1, 51):
    train_result = run_epoch(model, train_loader, criterion, device, optimizer=optimizer)
    scheduler.step()
    row = {'epoch': epoch, 'train_loss': train_result.loss, 'train_acc': train_result.accuracy}
    history.append(row)
    print(json.dumps(row, ensure_ascii=False))
final_train_acc = history[-1]['train_acc']
final_train_loss = history[-1]['train_loss']
final_test_acc, confusion, per_class_accuracy = evaluate_test(model, test_loader, device)
generalization_gap = final_train_acc - final_test_acc
model_path = artifact_dir / 'model_final.pt'
metrics_path = artifact_dir / 'metrics.json'
confusion_path = artifact_dir / 'confusion_matrix.npy'
class_accuracy_path = artifact_dir / 'class_accuracy.json'
torch.save({'model_state_dict': {k: v.cpu() for k, v in model.state_dict().items()}, 'feature_mean': mean, 'feature_std': std, 'final_train_acc': final_train_acc, 'final_train_loss': final_train_loss, 'test_acc': final_test_acc, 'generalization_gap': generalization_gap, 'per_class_accuracy': per_class_accuracy}, model_path)
metrics = {'final_train_acc': final_train_acc, 'final_train_loss': final_train_loss, 'test_acc': final_test_acc, 'generalization_gap': generalization_gap, 'per_class_accuracy': per_class_accuracy, 'epochs_ran': len(history), 'history': history}
metrics_path.write_text(json.dumps(metrics, indent=2, ensure_ascii=False), encoding='utf-8')
class_accuracy_path.write_text(json.dumps(per_class_accuracy, indent=2, ensure_ascii=False), encoding='utf-8')
np.save(confusion_path, confusion)
print('saved_to =', artifact_dir)


100%|██████████| 26.4M/26.4M [00:02<00:00, 10.5MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 185kB/s]
100%|██████████| 4.42M/4.42M [00:01<00:00, 3.14MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 5.02MB/s]


device = cuda
{"epoch": 1, "train_loss": 0.6570133598009745, "train_acc": 0.8509166666666667}
{"epoch": 2, "train_loss": 0.5607734377861023, "train_acc": 0.8866}
{"epoch": 3, "train_loss": 0.528154063129425, "train_acc": 0.9020666666666667}
{"epoch": 4, "train_loss": 0.49914510199228923, "train_acc": 0.9136666666666666}
{"epoch": 5, "train_loss": 0.47972690025965375, "train_acc": 0.9236166666666666}
{"epoch": 6, "train_loss": 0.46006361201604207, "train_acc": 0.93105}
{"epoch": 7, "train_loss": 0.4415339600245158, "train_acc": 0.9393833333333333}
{"epoch": 8, "train_loss": 0.42377523202896117, "train_acc": 0.94675}
{"epoch": 9, "train_loss": 0.40629704926808674, "train_acc": 0.9553666666666667}
{"epoch": 10, "train_loss": 0.3944556250890096, "train_acc": 0.9603833333333334}
{"epoch": 11, "train_loss": 0.3825651460488637, "train_acc": 0.9661666666666666}
{"epoch": 12, "train_loss": 0.36962829100290934, "train_acc": 0.97175}
{"epoch": 13, "train_loss": 0.3600873780250549, "train_acc": 0.

In [6]:
import json

metrics_path = artifact_dir / 'metrics.json'
metrics = json.loads(metrics_path.read_text(encoding='utf-8'))
print('final_train_acc =', metrics['final_train_acc'])
print('test_acc =', metrics['test_acc'])
print('generalization_gap =', metrics['generalization_gap'])
print('metrics_path =', metrics_path)

final_train_acc = 0.9999666666666667
test_acc = 0.906
generalization_gap = 0.09396666666666664
metrics_path = /content/drive/MyDrive/artifacts/fashion_mnist_ann_single_branch/metrics.json


In [7]:
!ls -lah "$artifact_dir"
!find "$artifact_dir" -maxdepth 1 -type f

total 11M
-rw------- 1 root root  142 Mar 25 19:12 class_accuracy.json
-rw------- 1 root root  928 Mar 25 19:12 confusion_matrix.npy
-rw------- 1 root root 5.6K Mar 25 19:12 metrics.json
-rw------- 1 root root  11M Mar 25 19:12 model_final.pt
/content/drive/MyDrive/artifacts/fashion_mnist_ann_single_branch/model_final.pt
/content/drive/MyDrive/artifacts/fashion_mnist_ann_single_branch/metrics.json
/content/drive/MyDrive/artifacts/fashion_mnist_ann_single_branch/class_accuracy.json
/content/drive/MyDrive/artifacts/fashion_mnist_ann_single_branch/confusion_matrix.npy
